# 🎙️ Golden Transcription Pipeline — Improved

## Training Data (3 sources, weighted by importance)

| Dataset | Rows | Language | Weight | Role |
|---------|------|----------|--------|------|
| FLEURS `validation_dataset.csv` | 692 | 20 languages | warm-start | Preserved in existing model checkpoint |
| `filtered_dataset.csv` (synthetic) | 470 | ar_eg | w=3 | Arabic synthetic training |
| `inputs/input1–49` | 49 | Arabic_SA | w=5 | Real labeled in-domain data (highest trust) |

**Test set (judged on):** `inputs/input50–99` (50 rows, Arabic_SA, no labels)

## Before running:
1. **GPU**: Settings → Accelerator → **GPU T4 x2** ✅
2. **Internet**: Settings → Internet → **ON** ✅
3. **Add Datasets** (right panel):
   - Your GitHub repo dataset containing `validation_dataset.csv`, `fleurs_audio/`, `filtered_dataset.csv`, `inputs/`
   - The `final_synthetic_audiio.zip` as a separate Kaggle dataset
4. Edit **Cell 3** with the correct paths
5. If you have `fusion_model.json` from a previous run, set `MODEL_R0` path — otherwise leave as None to train from scratch
6. Click **Run All**

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
import subprocess, sys
pkgs = ['openai-whisper','jiwer','xgboost','python-Levenshtein',
        'transformers','sentencepiece','accelerate','protobuf']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('✅ Packages installed')

In [ ]:
# ── Cell 2: Clone repo ─────────────────────────────────────────────────────
import subprocess, sys, os

REPO_URL = 'https://github.com/Vinar01/Team-team'
REPO     = '/kaggle/working/repo'   # ← used in all other cells

if not os.path.exists(REPO):
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO])
else:
    subprocess.check_call(['git', '-C', REPO, 'pull'])

print('✅ Repo ready at:', REPO)

# Show all input files so you can copy paths into Cell 3
print('\n── Your Kaggle input files ──')
for root, dirs, files in os.walk('/kaggle/input'):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    for f in files:
        print(' ', os.path.join(root, f))

In [ ]:
# ── Cell 3: ⚙️  EDIT THESE PATHS ──────────────────────────────────────────
# Copy paths printed by Cell 2 and paste here

# ── FLEURS TRAINING DATA ───────────────────────────────────────────────────
TRAIN_CSV   = '/kaggle/input/YOUR-DATASET/validation_dataset.csv'
TRAIN_AUDIO = '/kaggle/input/YOUR-DATASET/fleurs_audio'    # has en_us/, ar_eg/, ... subfolders

# ── SYNTHETIC ARABIC DATASET ───────────────────────────────────────────────
# filtered_dataset.csv is now in the GitHub repo (cloned above)
SYNTH_CSV      = f'{REPO}/filtered_dataset.csv'
# Path to the uploaded zip file of synthetic audio
SYNTH_AUDIO_ZIP = '/kaggle/input/YOUR-SYNTH-AUDIO-DATASET/final_synthetic_audiio.zip'
SYNTH_AUDIO_DIR = '/kaggle/working/synthetic_audio'   # where zip will be extracted

# ── ARABIC TEST DATA (100 rows — 50 unlabeled = what we're graded on) ─────
TEST_CSV    = '/kaggle/input/YOUR-DATASET/transcription_assessment.csv'
TEST_AUDIO  = '/kaggle/input/YOUR-DATASET/temp_audio'    # flat: 1.wav, 2.wav ...

# ── MODELS & OUTPUT ────────────────────────────────────────────────────────
# If you have fusion_model.json from a PREVIOUS Kaggle run, set the path here.
# The model will warm-start (fine-tune) from it — faster and better.
# If you don't have it, set to None and training starts from scratch.
MODEL_R0    = '/kaggle/working/fusion_model.json'   # previous FLEURS model (or None)
MODEL_FINAL = '/kaggle/working/model_final.json'    # new improved model
OUTPUT_PATH = '/kaggle/working/submission.csv'

# ── SETTINGS ───────────────────────────────────────────────────────────────
WHISPER_MODEL = 'small'    # small = better Arabic accuracy (still fits T4)
FAST_MODE     = False       # False = all 4 workflows (recommended)
LIMIT         = None        # Set e.g. 10 to test on 10 rows first

import torch
print(f'GPU: {"YES ✅" if torch.cuda.is_available() else "NO ❌ — go to Settings → Accelerator → GPU T4!"}')
print(f'Mode: {"FAST (Whisper+char)" if FAST_MODE else "FULL (Whisper+char+E5+mT5)"}')
print(f'Whisper: {WHISPER_MODEL}')
print(f'Warm-start from: {MODEL_R0}')

In [ ]:
# ── Cell 4: 🔨 Build combined training dataset ─────────────────────────────
# Extracts synthetic audio, builds arabic_labeled.csv, combines everything
import zipfile, pathlib, pandas as pd, os

# ── Step 1: Extract synthetic audio zip ────────────────────────────────────
print('Extracting synthetic audio...')
pathlib.Path(SYNTH_AUDIO_DIR).mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SYNTH_AUDIO_ZIP, 'r') as z:
    # Only extract actual wav files, skip macOS metadata
    wavs = [f for f in z.namelist() if f.endswith('.wav') and '__MACOSX' not in f]
    for f in wavs:
        z.extract(f, SYNTH_AUDIO_DIR)
print(f'  Extracted {len(wavs)} wav files to {SYNTH_AUDIO_DIR}')

# Find where ar_eg folder ended up
SYNTH_AUDIO_ROOT = None
for root, dirs, files in os.walk(SYNTH_AUDIO_DIR):
    if os.path.basename(root) == 'ar_eg':
        SYNTH_AUDIO_ROOT = str(pathlib.Path(root).parent)
        break
print(f'  Synthetic audio root: {SYNTH_AUDIO_ROOT}')

# ── Step 2: Fix synthetic CSV audio paths ──────────────────────────────────
# CSV has Windows paths like C:\...\fleurs_audio\ar_eg\498.wav
# We rewrite to fake Windows paths so _parse_windows_path() extracts ar_eg/{id}.wav
# and _find_audio() looks in {SYNTH_AUDIO_ROOT}/ar_eg/{id}.wav ✓
synth_df = pd.read_csv(SYNTH_CSV, encoding='utf-8-sig')
synth_df['audio'] = synth_df['audio_id'].apply(
    lambda aid: f'C:\\fake\\fleurs_audio\\ar_eg\\{aid}.wav'
)
synth_df['sample_weight'] = 3.0   # w=3: Arabic but synthetic/Egyptian dialect
# Keep correct_option (all 470 rows are labeled)
synth_df.to_csv('/kaggle/working/synthetic_fixed.csv', index=False)
print(f'\nSynthetic dataset: {len(synth_df)} rows (w=3, language=ar_eg)')
print(f'  correct_option distribution:\n{synth_df["correct_option"].value_counts().sort_index().to_string()}')

# ── Step 3: Build arabic_labeled.csv from inputs/input1-49 ─────────────────
# These are the 49 REAL labeled Arabic_SA rows — highest quality training data
INPUTS = pathlib.Path(f'{REPO}/inputs')
labeled_rows = []
for i in range(1, 50):
    d = INPUTS / f'input{i}'
    try:
        row = {
            'audio_id':    (d/'audio_id.txt').read_text().strip(),
            'language':    (d/'language.txt').read_text().strip(),
            'audio':       (d/'audio_url.txt').read_text().strip(),
            'option_1':    (d/'option_1.txt').read_text().strip(),
            'option_2':    (d/'option_2.txt').read_text().strip(),
            'option_3':    (d/'option_3.txt').read_text().strip(),
            'option_4':    (d/'option_4.txt').read_text().strip(),
            'option_5':    (d/'option_5.txt').read_text().strip(),
            'correct_option': (d/'correct_option.txt').read_text().strip(),
            'sample_weight': 5.0,   # w=5: real in-domain Arabic_SA data
        }
        labeled_rows.append(row)
    except Exception as e:
        print(f'  Warning: input{i} missing files: {e}')

labeled_df = pd.DataFrame(labeled_rows)
labeled_df.to_csv('/kaggle/working/arabic_labeled.csv', index=False)
print(f'\nArabic labeled (real): {len(labeled_df)} rows (w=5, language=Arabic_SA)')
print(f'  correct_option distribution:\n{labeled_df["correct_option"].value_counts().sort_index().to_string()}')

# ── Step 4: Combine all training data ──────────────────────────────────────
# NOTE: FLEURS (692 rows) is NOT re-scored — it's preserved via warm-start
# We only score the new Arabic data (~519 rows) which is ~45 min vs 3 hours
all_train = pd.concat([synth_df, labeled_df], ignore_index=True)
all_train.to_csv('/kaggle/working/all_arabic_train.csv', index=False)

print(f'\n{"="*60}')
print(f'COMBINED TRAINING SET: {len(all_train)} rows')
print(f'  Synthetic ar_eg:     {len(synth_df):3d} rows  (w=3)')
print(f'  Labeled Arabic_SA:   {len(labeled_df):3d} rows  (w=5)')
print(f'  FLEURS (via warmstart): 692 rows  (already in model checkpoint)')
print(f'{"="*60}')
print(f'\nSYNTH_AUDIO_ROOT = {SYNTH_AUDIO_ROOT!r}  ← used for synthetic audio')

In [ ]:
# ── Cell 5: 🚀 PHASE A — Train XGBoost on all Arabic data (warm-start) ─────
# Scores 519 Arabic rows (synthetic + labeled), fine-tunes XGBoost from
# existing FLEURS checkpoint — ~45 min instead of 3 hours from scratch
import subprocess, sys, os

cmd_train = [
    sys.executable, f'{REPO}/kaggle_runner.py',
    '--mode',         'train',
    '--train-csv',    '/kaggle/working/all_arabic_train.csv',
    '--train-audio',  SYNTH_AUDIO_ROOT,   # synthetic wav files live here
    '--model-path',   MODEL_FINAL,
    '--whisper-model', WHISPER_MODEL,
]
if not FAST_MODE:
    cmd_train.append('--full')
if MODEL_R0 and os.path.exists(MODEL_R0):
    cmd_train += ['--warmstart-model', MODEL_R0]
    print(f'Warm-starting from: {MODEL_R0}')
else:
    print('No previous model found — training from scratch')
if LIMIT:
    cmd_train += ['--limit', str(LIMIT)]

print('Running Phase A (Warm-start training on 519 Arabic rows)...')
print(' '.join(cmd_train))
print('=' * 60)

proc = subprocess.Popen(cmd_train, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()

if proc.returncode == 0 and os.path.exists(MODEL_FINAL):
    size = os.path.getsize(MODEL_FINAL) / 1024
    print(f'\n✅ Phase A done! Model saved ({size:.0f} KB): {MODEL_FINAL}')
else:
    print(f'\n❌ Phase A exited with code {proc.returncode}')

In [ ]:
# ── Cell 6: 🚀 PHASE B — Predict on Arabic test set (with structural filter) ─
# Whisper + char + E5 + mT5 scoring on 100-row test set, apply XGBoost + filter
# Structural filter kills 57KB screenplay options (length outlier + bracket count)
# Takes ~5-15 minutes on GPU T4
import subprocess, sys

cmd_pred = [
    sys.executable, f'{REPO}/kaggle_runner.py',
    '--mode',             'predict',
    '--test-csv',         TEST_CSV,
    '--test-audio',       TEST_AUDIO,
    '--model-path',       MODEL_FINAL,
    '--output',           OUTPUT_PATH,
    '--whisper-model',    WHISPER_MODEL,
    '--structural-filter',   # post-processing: kills screenplay/outlier options
]
if not FAST_MODE:
    cmd_pred.append('--full')
if LIMIT:
    cmd_pred += ['--limit', str(LIMIT)]

print('Running Phase B (Prediction on test set with structural filter)...')
print(' '.join(cmd_pred))
print('=' * 60)

proc = subprocess.Popen(cmd_pred, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()

if proc.returncode == 0:
    print('\n✅ Phase B done!')
else:
    print(f'\n❌ Phase B exited with code {proc.returncode}')

In [ ]:
# ── Cell 7: 📊 View results + accuracy on 49 labeled rows ──────────────────
import pandas as pd, pathlib

results = pd.read_csv(OUTPUT_PATH)
print(f'Output: {len(results)} rows  |  {OUTPUT_PATH}')
print()

# ── Show prediction table ───────────────────────────────────────────────────
cols = ['audio_id', 'language', 'predicted_option_num',
        'score_1', 'score_2', 'score_3', 'score_4', 'score_5']
print(results[[c for c in cols if c in results.columns]].head(20).to_string(index=False))

# ── Accuracy on 49 labeled rows using correct_options.txt ──────────────────
# correct_options.txt in the repo has the ground-truth labels for inputs 1-49
correct_file = pathlib.Path(f'{REPO}/inputs/correct_options.txt')
if correct_file.exists():
    lines = correct_file.read_text().strip().split('\n')
    # Build dict: audio_id (1-based) → correct option number
    correct_map = {}
    for i, line in enumerate(lines):
        val = line.strip()
        if val.isdigit():
            correct_map[i + 1] = int(val)

    ok, total = 0, 0
    for _, row in results.iterrows():
        try:
            aid = int(row['audio_id'])
        except (ValueError, KeyError):
            continue
        if aid in correct_map:
            total += 1
            if int(row['predicted_option_num']) == correct_map[aid]:
                ok += 1

    print(f'\n{"="*55}')
    if total > 0:
        pct = 100 * ok / total
        print(f'ACCURACY ON {total} LABELED ROWS: {ok}/{total}  ({pct:.1f}%)')
        print(f'Previous best:                    22/49  (44.9%)')
        delta = pct - 44.9
        sign  = '+' if delta >= 0 else ''
        print(f'Change:                          {sign}{delta:.1f}%')
    else:
        print('No labeled rows found in results — check audio_id column')
    print(f'{"="*55}')
else:
    print('(correct_options.txt not found — skipping accuracy check)')

# ── Structural filter sanity check ─────────────────────────────────────────
score_cols = ['score_1', 'score_2', 'score_3', 'score_4', 'score_5']
sc = results[score_cols]
print(f'\nScore stats (structural filter should raise confidence):')
print(f'  Avg max score  : {sc.max(axis=1).mean():.3f}  (was ~0.22 — higher = more confident)')
print(f'  Avg entropy    : {(-sc * sc.clip(lower=1e-9).apply(lambda x: x.map(__import__("math").log))).sum(axis=1).mean():.3f}  (lower = less entropy)')

# ── Predicted option distribution ──────────────────────────────────────────
print(f'\nPredicted option distribution:')
print(results['predicted_option_num'].value_counts().sort_index().to_string())

print('\n✅ Download submission.csv from the Output tab →')